# Pure Press: retrain our own pair model on Colab

Retrains the small multilingual model the server ships (`backend/models/minilm-l12-onnx-int8`) with our own
data added: Google News headlines judged against forwarded claims (labelled by Gemini through the
verifier's stance step), on top of the public sets it learned from before (MNLI, FEVER, FNC-1, XNLI Hindi,
IndicXNLI Tamil and Hindi, our synthetic pairs, the claim-matching suites).

**Before you start:** Runtime → Change runtime type → **T4 GPU** (any GPU works). Then Runtime → **Run all**.
When asked, upload `pure_press_ml.zip`. About 35–45 minutes in all; the last cell downloads
`pair_model_export.zip` (about 45 MB). Put that file in the project's `ml/` folder and tell Claude.

Nothing here touches the project or the server; the result is checked against the shipped model before it is used.

In [ ]:
# 1. A GPU is required
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.'
print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())

In [ ]:
# 2. Upload pure_press_ml.zip (the training scripts, our labelled pairs and the shipped model for comparison)
import io, zipfile
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
zipfile.ZipFile(io.BytesIO(uploaded[name])).extractall('/content/pp')
%cd /content/pp
!ls

In [ ]:
# 3. Libraries (torch is already on Colab)
!pip install -q "transformers>=5" datasets accelerate scikit-learn sentencepiece protobuf onnx onnxruntime onnxscript huggingface_hub pandas
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 4. Data: the public sets, our synthetic pairs, then train / dev / evaluation files (about 5 minutes)
!python -X utf8 ml/src/fetch_data.py ml/data/raw
!python -X utf8 ml/src/synth.py --out ml/data
!cp stance_pairs.jsonl ml/data/
!python -X utf8 ml/src/build_data.py ml/data backend/eval/suites

In [ ]:
# 5. Fine-tune (the recipe of the model we ship: 2 epochs, lr 3e-5; about 20-25 minutes on a T4)
!python -X utf8 ml/src/train.py --model MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli --data ml/data --out ml/artifacts/minilm-l12-ft --epochs 2 --lr 3e-5

In [ ]:
# 6. Export to ONNX with a pruned vocabulary and int8 weights, as the server loads it
!python -X utf8 ml/src/export_onnx.py --model ml/artifacts/minilm-l12-ft --out ml/artifacts/minilm-l12-onnx-int8 --data ml/data

In [ ]:
# 7. Measure: the new int8 model against the one the server ships, on the same held-out sets
!mkdir -p ml/artifacts/eval
!python -X utf8 ml/src/evaluate.py --model shipped_model --onnx shipped_model/onnx/model_quantized.onnx --data ml/data --out ml/artifacts/eval/shipped_int8.json
!python -X utf8 ml/src/evaluate.py --model ml/artifacts/minilm-l12-onnx-int8 --onnx ml/artifacts/minilm-l12-onnx-int8/onnx/model_quantized.onnx --data ml/data --out ml/artifacts/eval/new_int8.json
!python -X utf8 ml/src/tables.py "shipped=ml/artifacts/eval/shipped_int8.json" "new=ml/artifacts/eval/new_int8.json" | tee ml/artifacts/eval/comparison.md

In [ ]:
# 8. Download the export and the measurements (about 45 MB)
!cp ml/artifacts/minilm-l12-ft/train_log.json ml/artifacts/eval/ 2>/dev/null; true
!cd ml/artifacts && rm -f /content/pair_model_export.zip && zip -qr /content/pair_model_export.zip minilm-l12-onnx-int8 eval -x 'minilm-l12-onnx-int8/onnx/model.onnx'
!ls -la /content/pair_model_export.zip
files.download('/content/pair_model_export.zip')